# 3. 모델 학습

변수를 고르고 모델을 학습한다. 성능 평가는 `4-모델성능평가` 에서 다룬다.

변수는 **세 단계**로 좁혔다. 각 단계가 앞 단계만으로는 알 수 없는 것을 걸러낸다.

| 단계 | 방법 | 결과 |
|---|---|---|
| 1 | 타겟과의 상관계수 | 9개 → 2개 |
| 2 | 시간순 검증 | 2개 → 1개 |
| 3 | 대조 실험 | 우연이 아님을 확인 |

In [ ]:
import sys, os, warnings
sys.path.insert(0, '/data/Project/boyoung/pipeline')
for v in ('OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS'):
    os.environ.setdefault(v, '1')
warnings.filterwarnings('ignore')

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
plt.rc('font', family='NanumGothic'); plt.rc('axes', unicode_minus=False)
sns.set_theme(style='whitegrid', font='NanumGothic', rc={'axes.unicode_minus': False})
pd.set_option('display.width', 180); pd.set_option('display.max_columns', 40)

import db, preprocess as P, classify as C

data, report = P.build()
data = data.sort_values('target_date').reset_index(drop=True)
feats = P.feature_columns(data)
th = C.threshold(data['bad_rate'])
y  = C.label(data['bad_rate'], th)
yv = y.values
POS = [n for n in range(25, len(yv)) if yv[:n].sum() >= 25]   # 학습 양성 25건 이상

print(f'{len(data)}로트  양성 {int(yv.sum())}  기준선 {C.baseline_accuracy(y):.3f}  집계 분할 {len(POS)}')

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score

def run(X, yy=None, model='xgboost', weight='balanced'):
    """시간순 검증. 학습 구간만 보고 임계값을 정한다."""
    yy = yv if yy is None else yy
    v = []
    for n in POS:
        tr, te = slice(0, n), slice(n, None)
        if yy[te].sum() < 1 or len(set(yy[tr])) < 2:
            continue
        cut = C.cut_rule(X.iloc[tr], yy[tr], weight=weight, model=model)
        m = make_pipeline(C.impute(X.columns), C.candidates(0, weight)[model])
        m.fit(X.iloc[tr], yy[tr])
        p = (m.predict_proba(X.iloc[te])[:, 1] >= cut).astype(int)
        v.append((accuracy_score(yy[te], p), f1_score(yy[te], p, zero_division=0),
                  recall_score(yy[te], p, zero_division=0), precision_score(yy[te], p, zero_division=0)))
    a = np.array(v)
    return dict(정확도=a[:,0].mean(), F1=a[:,1].mean(), F1편차=a[:,1].std(),
                재현율=a[:,2].mean(), 정밀도=a[:,3].mean())

## 3.1 1단계 — 상관계수로 후보를 좁힌다

In [ ]:
MM = {'벤토나이트':'bentonite', '모래온도':'sand_temp_mean'}
MM |= {n: f'{p}_val_mean' for p, n in P.MEASURES if p != 'sand_temp'}
corr = pd.Series({n: abs(data[c].rank().corr(y.rank())) for n, c in MM.items()}).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8.5, 3.4))
sns.barplot(x=corr.index, y=corr.values, ax=ax, hue=(corr.values > .3), legend=False,
            palette={True:'#b8531c', False:'#cfe0ea'})
ax.axhline(.3, color='#333', ls='--', lw=1.2); ax.set_ylabel('불량 판정과의 순위 상관')
plt.setp(ax.get_xticklabels(), rotation=20, ha='right')
ax.set_title('9개 중 2개만 0.3 을 넘는다')
plt.tight_layout(); plt.show()
print(corr.round(3).to_string())

규격 기준이 **있는** 일곱 항목은 모두 0.07 미만이고, 기준이 **없는** 두 항목만 0.3 을 넘는다.

## 3.2 2단계 — 시간순 검증으로 고른다

상관계수는 후보를 좁히지만 **둘 중 어느 쪽인지는 알려주지 못한다.** 실제로 순위가 뒤집힌다.

In [ ]:
rows = []
for n, cols in [('벤토나이트', ['bentonite']), ('모래온도', ['sand_temp_mean']),
                ('벤토 + 모래온도', ['bentonite','sand_temp_mean'])]:
    rows.append({'구성': n, **run(data[cols])})
print(pd.DataFrame(rows).round(3).to_string(index=False))
print()
order = pd.Series(np.arange(len(data))).rank()
for n, c in [('벤토나이트','bentonite'), ('모래온도','sand_temp_mean')]:
    print(f'  {n} ↔ 로트 순서  {data[c].rank().corr(order):+.3f}')

**모래온도는 사실상 달력이다.** 로트 순서와 `+0.867` 로 붙어 있다.

시간순 검증에서는 학습 구간과 검증 구간의 온도대가 다르다. 1~7월로 배운 모델이 8~9월을 만나면 **학습에서 거의 보지 못한 고온 구간**이라 전부 정상이라고 답해 버린다.

무작위로 나누면 이 문제가 보이지 않는다. **상관계수와 무작위 검증만으로는 놓치는 함정이다.**

## 3.3 3단계 — 대조 실험

탐색 결과가 아니라 **대조군**으로 확인한다.

### ① 값의 순서에 정보가 있는가

`28/1` 이 배합 코드 하나라면 28 이라는 숫자에 크기의 의미가 없다.

In [ ]:
vals = sorted(data['bentonite'].dropna().unique())
rng = np.random.default_rng(0)
onehot = pd.get_dummies(data['bentonite'].astype('Int64').astype(str),
                        prefix='b', dummy_na=True).astype(float)

rows = [{'취급':'숫자 그대로', **run(data[['bentonite']])}]
sh = [run(data[['bentonite']].assign(
          bentonite=data['bentonite'].map(dict(zip(vals, np.random.default_rng(s).permutation(vals))))))
      for s in range(5)]
rows.append({'취급':'순서 뒤섞기 (5회 평균)',
             '정확도':np.mean([x['정확도'] for x in sh]), 'F1':np.mean([x['F1'] for x in sh]),
             'F1편차':np.std([x['F1'] for x in sh]),
             '재현율':np.mean([x['재현율'] for x in sh]), '정밀도':np.mean([x['정밀도'] for x in sh])})
rows.append({'취급':'원-핫 (순서 무시)', **run(onehot)})
print(pd.DataFrame(rows).round(3).to_string(index=False))
print()
print('  뒤섞기 5회 F1:', np.round([x['F1'] for x in sh], 3))

**순서를 없애면 무너진다.** 코드였다면 순서를 바꿔도 성능이 유지되어야 한다.

### ② 우연으로 나올 수 있는 성능인가

타겟을 무작위로 섞어 변수와의 연결을 끊고 같은 절차를 반복한다.

In [ ]:
real = run(data[['bentonite']])
rng = np.random.default_rng(0)
perm = [run(data[['bentonite']], rng.permutation(yv)) for _ in range(60)]
pa = np.array([x['정확도'] for x in perm]); pf = np.array([x['F1'] for x in perm])

fig, ax = plt.subplots(figsize=(7.5, 3.4))
sns.histplot(pa, bins=14, ax=ax, color='#cfe0ea', label=f'타겟을 섞었을 때 ({len(pa)}회)')
ax.axvline(real['정확도'], color='#b8531c', lw=2.4, label=f'실제 {real["정확도"]:.3f}')
ax.set_xlabel('정확도'); ax.legend(fontsize=9)
ax.set_title(f'{len(pa)}회 중 {int((pa >= real["정확도"]).sum())}회만 실제를 넘었다')
plt.tight_layout(); plt.show()
print(f'정확도 초과 {(pa >= real["정확도"]).mean():.3f}   F1 초과 {(pf >= real["F1"]).mean():.3f}')

### ③ 변수를 더하면 좋아지는가

의미 없는 난수를 더해 본다. **정보가 없는데도 F1 이 오른다면, 그 상승은 측정이 흔들려서 생긴 것**이다.

In [ ]:
rng = np.random.default_rng(0)
noise = []
for _ in range(30):
    X = data[['bentonite']].copy(); X['noise'] = rng.normal(size=len(data))
    noise.append(run(X)['F1'])
noise = np.array(noise)

print(f'벤토나이트 단독            F1 {real["F1"]:.3f}')
print(f'난수를 하나 더했을 때 (30회)  F1 평균 {noise.mean():.3f}  최대 {noise.max():.3f}')
print(f'  단독보다 오른 경우 {int((noise > real["F1"]).sum())}/30 = {(noise > real["F1"]).mean():.0%}')
print()
print('실제 변수 30개를 하나씩 더했을 때는 23% 에서만 올랐다.')
print('→ 잡음이 더 자주 오른다. 변수 추가로 얻는 개선은 우연과 구분되지 않는다.')

## 3.4 변수를 하나만 쓰는 근거

In [ ]:
print(f'양성 {int(yv.sum())}건.  변수당 양성 10건 기준으로 최대 {int(yv.sum())//10}개\n')
print('벤토나이트는 8종 이산값이라 이미 8그룹이다. 변수를 더하면 곱셈으로 쪼개진다.\n')
rows = [{'구성':'벤토나이트 단독', '그룹':int(data['bentonite'].nunique()),
         '그룹당 로트':round(len(data)/data['bentonite'].nunique(),1), '1~2개뿐인 그룹':0}]
for c, n in [('perm_val_mean','+ 통기도'), ('night_ratio','+ 야간비율')]:
    gg = data.assign(b=pd.qcut(data[c].rank(method='first'), 3)).groupby(['bentonite','b'], observed=True).size()
    rows.append({'구성':f'{n} (3구간)', '그룹':len(gg),
                 '그룹당 로트':round(len(data)/len(gg),1), '1~2개뿐인 그룹':int((gg<=2).sum())})
gg = data.assign(b=pd.qcut(data['perm_val_mean'].rank(method='first'),3),
                 c=pd.qcut(data['night_ratio'].rank(method='first'),3)).groupby(['bentonite','b','c'], observed=True).size()
rows.append({'구성':'+ 둘 다', '그룹':len(gg), '그룹당 로트':round(len(data)/len(gg),1),
             '1~2개뿐인 그룹':int((gg<=2).sum())})
print(pd.DataFrame(rows).to_string(index=False))

**그룹당 로트가 1~2개면 그 그룹의 판정이 로트 한두 개로 정해진다** — 학습이 아니라 암기다.

## 3.5 후보 모델 — BADA 구성을 그대로

BADA 솔루션의 분류 후보를 가져왔다. 같은 모델 목록으로 학습해 비교할 수 있게 하기 위해서다.

| 파일 | 내용 |
|---|---|
| `infrastructure/constants/model_config.py:18` | `ENSEMBLE_MODEL_MAP` — 후보 7종 |
| `infrastructure/constants/hyperparameters_config.py` | 탐색 범위 |
| `service/ai_engine/model/ensemble.py:111` | 평가 지표 |
| `service/ai_engine/model/ensemble.py:159` | `KFold(shuffle=True)` |

**모델 목록은 그대로 두고 설정만 바꿨다.** BADA 의 탐색 범위는 수천 건 규모를 전제하므로, 표본 93 에 그대로 적용하면 전부 암기가 된다.

| 항목 | BADA | 이 프로젝트 | 이유 |
|---|---|---|---|
| `max_depth` | 5, 10, 15, None | **2~3** | 깊이 10 이면 93개를 다 외운다 |
| `class_weight` | 설정 없음 | **`balanced`** | 없으면 시간순 재현율 0 |
| 판정 임계값 | 0.5 고정 | **규칙으로 산출** | 0.5 로 자르면 F1 0.053 |
| 교차검증 | `KFold(shuffle=True)` | **`StratifiedKFold`** | 폴드마다 양성 비율을 맞춘다 |
| 주 검증 | 무작위 | **시간순** | 실제 운용 조건 |
| 지표 | `f1_weighted` | **`f1` (양성 기준)** | weighted 는 0.1 후하다 |
| 결측 대체 | 전체 중앙값 | **분할 안에서 계산** | 전체 중앙값은 누수 |

In [ ]:
import inspect
print(inspect.getsource(C.candidates))

## 3.6 모델 비교

In [ ]:
rows = []
for m in ['xgboost','random forest','logistic regression','decision tree',
          'lgbm','gradient boosting','adaboost']:
    try: rows.append({'모델': m, **run(data[['bentonite']], model=m)})
    except Exception: pass
print(pd.DataFrame(rows).sort_values('F1', ascending=False).round(3).to_string(index=False))

**`xgboost` 를 택한다.** 로지스틱은 재현율이 높지만 거의 전부를 불량으로 찍어 정확도가 무너진다.

`linear svm` 은 확률을 내지 않아 임계값 방식과 맞지 않으므로 제외했다.

## 3.7 판정 임계값

양성이 34% 뿐이라 확률이 0.5 를 거의 넘지 않는다. 검증 성적을 보고 고르면 누수이므로, **학습 구간의 양성 비율**에서 산출한다.

In [ ]:
cut = C.cut_rule(data[['bentonite']][:70], y[:70])
m = make_pipeline(C.impute(['bentonite']),
                  C.candidates(0,'balanced')['xgboost']).fit(data[['bentonite']][:70], y[:70])
prob = m.predict_proba(data[['bentonite']][70:])[:, 1]

fig, ax = plt.subplots(figsize=(7.5, 3.2))
sns.histplot(prob, bins=16, ax=ax, color='#7aa6c2')
ax.axvline(cut, color='#b8531c', lw=2.2, label=f'산출된 임계값 {cut:.3f}')
ax.axvline(.5, color='#333', ls='--', lw=1.4, label='기본값 0.5')
ax.set_xlabel('모델이 낸 확률'); ax.legend(fontsize=9); ax.set_title('확률이 0.5 를 넘지 않는다')
plt.tight_layout(); plt.show()
print(inspect.getsource(C.cut_rule))

## 3.8 최종 구성

| | |
|---|---|
| 변수 | `bentonite` 1개 |
| 모델 | `xgboost` · `max_depth=2` · `class_weight='balanced'` |
| 임계값 | 학습 구간 양성비율에서 산출 |
| 결측 | 측정값 중앙값 · 규격이탈 0 · 분할 안에서 계산 |

성능 평가는 `4-모델성능평가` 참조.